# Surplus App â€” Data Visualization

Exploration notebook for the data and calculations in `analysis.py`.
Nothing here is deployed â€” the Dash app never touches this file.

**Run the import cell below first** (it downloads the CSVs once per kernel
session). After that, every dataframe and function is available in any cell,
in any order, without reloading.

In [1]:
# Auto-reload edited .py modules before each cell run, so changes to
# analysis.py are picked up without restarting the kernel.
# Note: functions (update_mean, update_graph) refresh automatically, but the
# imported dataframes are snapshots â€” re-run the import cell below to refresh
# them after an edit (this also re-downloads the CSVs once per change).
# %load_ext autoreload
# %autoreload 2

In [2]:
import sys
from pathlib import Path

# The notebook lives in notebooks/ but the app modules live in src/, so put
# src/ on the import path. Walking up from the working directory (instead of
# hardcoding '..') keeps this working whether the kernel starts in notebooks/
# or at the project root.
src_dir = next(p / 'src' for p in [Path.cwd(), *Path.cwd().parents]
               if (p / 'src' / 'analysis.py').exists())
sys.path.insert(0, str(src_dir))

import pandas as pd

from analysis import (temp, efport, df_combined, df_combined_10, bins_df,
                      df_lirr, df_lirr_output, vol_list, bin_yaxis_values,
                      update_mean, update_graph)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print(f'Loaded: temp {temp.shape}, efport {efport.shape}, '
      f'df_combined {df_combined.shape}, df_combined_10 {df_combined_10.shape}, '
      f'bins_df {bins_df.shape}')

Loaded: temp (5000, 3), efport (22, 4), df_combined (20000, 3), df_combined_10 (20000, 3), bins_df (21, 23)


## Source data

`efport` is the efficient frontier (22 points). `temp` holds the Monte Carlo
simulated portfolios; `df_combined` / `df_combined_10` are the 1-year and
5-year return simulations; `bins_df` is the liability PDF surface.

In [3]:
# This corresponds to the basic efficient frontier chart

efport

,targetrets,targetvols,targetsharpe,type
0,4.30,3.04,1.42,efficient frontier
1,4.55,3.12,1.46,efficient frontier
2,4.80,3.31,1.45,efficient frontier
3,5.04,3.59,1.41,efficient frontier
4,5.29,3.93,1.35,efficient frontier
5,5.54,4.38,1.26,efficient frontier
6,5.79,5.13,1.13,efficient frontier
7,6.03,6.09,0.99,efficient frontier
8,6.28,7.19,0.87,efficient frontier
9,6.53,8.37,0.78,efficient frontier


In [4]:
# This all appears to be related to the extended 10 year efficient frontiers

display(temp.head())
display(df_combined.head())
display(df_combined_10.head())
display(bins_df.head())

,port_rets,port_vols,sharpe_ratio
0,6.77,9.69,0.70
1,6.68,10.62,0.63
2,7.07,12.93,0.55
3,7.10,12.23,0.58
4,8.34,18.23,0.46


,r,x,Z
0,0.047374,1,0.027566
1,0.036085,1,0.365912
2,0.055768,1,0.223986
3,0.024531,1,0.712171
4,0.084155,1,1.074737


,r,x,Z
0,0.277622,1,0.173633
1,0.220868,1,0.600713
2,0.331927,1,0.914571
3,0.216118,1,0.665531
4,0.263463,1,0.019551


,Unnamed: 0,0.0,0.5,1.0,1.5,2.0,2.5,3.0,3.5,4.0,4.5,5.0,5.5,6.0,6.5,7.0,7.5,8.0,8.5,9.0,9.5,10.0,10.5
0,-0.114051,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23
1,-0.096751,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16
2,-0.079451,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32
3,-0.062151,57,57,57,57,57,57,57,57,57,57,57,57,57,57,57,57,57,57,57,57,57,57
4,-0.044851,66,66,66,66,66,66,66,66,66,66,66,66,66,66,66,66,66,66,66,66,66,66


## Liability assumptions (Step 1 page tables)

In [5]:
display(df_lirr)
display(df_lirr_output)

,,Starting Wealth,Cash Outflow Yr 1,Cash Outflow Yr 2,Cash Outflow Yr 3,Cash Outflow Yr 4,Wealth Bequest Yr 5
0,Cash Flow,"$1,000,000","-$50,000","-$50,000","-$50,000","-$50,000","-$1,050,000"
1,Standard Deviation,0%,25%,25%,25%,25%,25%


,Liability Discount Rate (%),Liability Standard Deviation (σ)
0,5,6


## Surplus table

Change the inputs and re-run this cell to see the full `update_mean` output
for any liability assumption â€” including values the app's dropdowns don't offer.

In [6]:
discount_rate = 5.0   # mean liability discount rate (%)
sigma = 6.0           # spending flexibility / liability std dev (%)

# target vols corresponds to the efficient frontier
# arithmetic mean is the efficient frontier minus the simple 5% liability discount rate
# Mean surplus only differs in that it takes the compounded asset and liability returns over a time horizon and then annualizes them, which affects the surplus results given EF has less volatility over time when compounded (with constant drift)

surplus_df = update_mean(discount_rate, sigma)

# Display-only reformat: promote the shared 'Output' prefix on the last four
# columns to a second header level that spans them. The app and update_graph
# still see the flat single-level names update_mean returns.
surplus_df.columns = pd.MultiIndex.from_tuples(
    [('Output', c.removeprefix('Output ')) if c.startswith('Output ')
     else ('', c) for c in surplus_df.columns])

# Style the promoted 'Output' header: centered over its four columns, with a
# line underneath spanning them. Pandas renders the spanning cell as
# th.col_heading.level0.col<N> where N is the group's first column position,
# so find that position instead of hardcoding it.
first_output_col = next(i for i, c in enumerate(surplus_df.columns)
                        if c[0] == 'Output')
surplus_df.style.set_table_styles([
    {'selector': f'th.col_heading.level0.col{first_output_col}',
     'props': [('text-align', 'center'),
               ('border-bottom', '2px solid currentColor')]},
])

## Interactive figures

The same two figures the app's Outputs page builds, for the inputs chosen above.

In [7]:
# Efficent frontier is same as original basic line
# Mean liability discount rate = the basic 5% shown on the illustrative table and page
# The "Arithmetic Mean Surplus" line = the Arithmetic Mean Surplus column in the df

# You can see that in your table: the column bottoms out at ~0.0009 in row 7,
# where the portfolio vol (6.09%) nearly equals your sigma input (6%), and
# grows as vol moves away from 6% in either direction. So BE Surplus_95% is
# 1.65 standard deviations of surplus risk â€” the buffer the portfolio must
# clear at 95% confidence â€” and it's why the risk-adjusted optimum lands near
# the frontier point whose volatility matches the liability's.
surplus_fig, utility_fig = update_graph(discount_rate, sigma)

# Overlay the 95% break-even surplus hurdle (z=1.65 x surplus std dev).
# The figure's y-axis is in percent, so plot the 'Output BE (95% BE x 100)'
# column, which is 'BE Surplus_95%' x 100.
surplus_table = update_mean(discount_rate, sigma)
# surplus_fig.add_scatter(x=surplus_table['targetvols'],
#                         y=surplus_table['Output BE (95% BE x 100)'],
#                         mode='lines', name='BE Surplus 95%',
#                         line=dict(color='purple', dash='dash'))

# Overlay the compounded-and-annualized mean surplus, in percent, with a
# text label at the line's right end (this figure's legend is hidden, so
# each curve is identified by an annotation instead)
surplus_fig.add_scatter(x=surplus_table['targetvols'],
                        y=surplus_table['Output Mean Surplus (x 100)'],
                        mode='lines', name='Mean Surplus over Time Horizon',
                        line=dict(color='crimson', dash='dash'))
_last = surplus_table.iloc[-1]
surplus_fig.add_annotation(x=_last['targetvols'],
                           y=_last['Output Mean Surplus (x 100)'],
                           text='Mean Surplus over Time Horizon',
                           showarrow=False,
                           xanchor='right', yanchor='bottom', yshift=3,
                           font=dict(size=10, color='crimson'))

# Overlay the risk premium in percent â€” the 95% confidence buffer in excess
# of the mean surplus; it bottoms out where portfolio vol matches the
# liability sigma and grows steeply toward both ends of the frontier
surplus_fig.add_scatter(x=surplus_table['targetvols'],
                        y=surplus_table['Output Risk Premium (x 100)'],
                        mode='lines', name='Risk Premium',
                        line=dict(color='teal', dash='dash'))
surplus_fig.add_annotation(x=_last['targetvols'],
                           y=_last['Output Risk Premium (x 100)'],
                           text='Risk Premium', showarrow=False,
                           xanchor='right', yanchor='bottom', yshift=3,
                           font=dict(size=10, color='teal'))
surplus_fig.show()

In [8]:
utility_fig.show()

## Scratch

Experiment freely below â€” nothing in this notebook affects the deployed app.